# DEMO 3: Advanced Interactivity Across Data Types

This demo builds on Demos 1 & 2. We’ll add **interactivity** to dashboards and explore how each data architecture handles dynamic user input differently.

Topics covered:
1. **Filter widgets** — interactive slicers for viewers
2. **Query parameters** — dynamic SQL placeholders injected at runtime
3. **Cross-filtering** — click a data point to filter other widgets
4. **Parameterized Metric Views** — metric views with built-in parameters
5. **Best practices** — which interactivity pattern works best with which data type

### How Interactivity Differs by Data Architecture

| Feature | Star Schema | Gold Table | SQL View | Metric View |
| --- | --- | --- | --- | --- |
| Filter widgets | Works on any column | Works on any column | Limited to output columns | Works on dimensions |
| Parameters | Full flexibility | Full flexibility | Must parameterize the view | Built-in parameter support |
| Cross-filtering | Needs shared column names | Natural (flat) | Limited | Automatic on dimensions |
| Consistent definitions | No | No | Partial | Yes (MEASURE enforced) |

> **Prerequisite:** Run the Setup notebook first. Open your dashboard in draft mode to follow along.

---

## Part 1: Filter Widgets — How They Interact with Each Data Type

### Filter Scope Recap
| Scope | Who Controls | Applies To | Power BI Equivalent |
| --- | --- | --- | --- |
| **Global** | Viewer | ALL pages, all widgets | Report-level filter |
| **Page-level** | Viewer | All widgets on ONE page | Page-level filter |
| **Widget-level** | Author (hidden) | ONE widget only | Visual-level filter |

### How Filters Work with Each Data Architecture

**Star Schema / Gold Table:**
- Filters operate on **columns in your dataset query results**
- If your query returns `region`, `channel`, `product_category`, viewers can filter on any of these
- The filter injects a WHERE clause into the dataset SQL

**SQL Views:**
- Filters operate on the **output columns of the view**
- If `vw_region_performance` returns `region_name` and `territory`, those are filterable
- ⚠️ If you need to filter on a column NOT in the view output, you must modify the view

**Metric Views:**
- Filters operate on **dimensions** defined in the metric view YAML
- Only dimensions are filterable (measures cannot be filtered directly)
- Benefit: the dimension list is curated and labeled — viewers see clean names

### Best Practice for Filters
| Data Type | Filter Strategy |
| --- | --- |
| Star Schema | Include all likely filter columns in SELECT |
| Gold Table | All columns available automatically |
| SQL View | Ensure the view output includes filter columns |
| Metric View | Dimensions are the natural filter targets |

---

In [0]:
%sql
-- ============================================================
-- PART 2: QUERY PARAMETERS — Dynamic SQL for Dashboards
-- ============================================================
-- Parameters inject values directly into SQL at runtime.
-- Syntax: :parameter_name (colon prefix)
-- The dashboard creates a parameter widget for viewers to fill.

-- STAR SCHEMA with parameter: Revenue by Category, filtered by Region
SELECT
  p.category                     AS product_category,
  ROUND(SUM(f.net_revenue), 2)   AS total_revenue,
  COUNT(f.order_id)              AS order_count,
  ROUND(AVG(f.net_revenue), 2)   AS avg_order_value
FROM fact_sales f
JOIN dim_product  p ON f.product_id  = p.product_id
JOIN dim_region   r ON f.region_id   = r.region_id
WHERE r.region_name = :selected_region
GROUP BY p.category
ORDER BY total_revenue DESC;

-- GOLD TABLE with parameter (simpler):
-- SELECT
--   product_category,
--   ROUND(SUM(net_revenue), 2) AS total_revenue,
--   COUNT(order_id)            AS order_count
-- FROM gold_sales
-- WHERE region = :selected_region
-- GROUP BY product_category
-- ORDER BY total_revenue DESC;

In [0]:
%sql
-- ============================================================
-- PARAMETERS WITH SQL VIEWS
-- ============================================================
-- Views don't natively accept parameters, but you can use parameters
-- in the dashboard query that references the view.

-- Option A: Filter the view output (works for simple cases)
SELECT
  order_month,
  region,
  total_revenue,
  total_profit,
  order_count
FROM vw_monthly_revenue
WHERE region = :selected_region
ORDER BY order_month;

-- Option B: Use a parameterized query against the star schema
-- when the view doesn't expose the filter you need
-- SELECT
--   DATE_TRUNC('MONTH', f.order_date) AS order_month,
--   ROUND(SUM(f.net_revenue), 2) AS total_revenue
-- FROM fact_sales f
-- JOIN dim_region r ON f.region_id = r.region_id
-- WHERE r.territory = :selected_territory
-- GROUP BY DATE_TRUNC('MONTH', f.order_date)
-- ORDER BY order_month;

In [0]:
%sql
-- ============================================================
-- PARAMETERS WITH METRIC VIEWS
-- ============================================================
-- Metric views support parameters natively in YAML!
-- The dashboard passes values directly into the metric view definition.

-- Querying the metric view with a dashboard parameter:
-- The WHERE clause filters on the metric view's dimensions.
SELECT
  `Product Category`,
  MEASURE(`Total Revenue`)   AS total_revenue,
  MEASURE(`Order Count`)     AS order_count,
  MEASURE(`Avg Order Value`) AS avg_order_value,
  MEASURE(`Profit Margin`)   AS profit_margin
FROM mv_sales_metrics
WHERE `Region` = :selected_region
GROUP BY ALL
ORDER BY total_revenue DESC;

-- NOTE: Metric views also support YAML-level parameters for
-- more advanced use cases (thresholds, date ranges, etc.).
-- Example: mv_sales_metrics(min_amount => 100, region => 'Northeast')

## Part 3: When to Use Filters vs Parameters

### Quick Decision Guide

| Scenario | Use Filter Widget | Use Parameter |
| --- | --- | --- |
| Filter on a column in the result set | ✓ | |
| Filter on a value NOT in the result set | | ✓ |
| Need CASE/conditional logic based on selection | | ✓ |
| Dynamic date ranges (last N days) | | ✓ |
| Simple dropdown of existing values | ✓ | |
| Same filter applied across multiple datasets | | ✓ (shared parameter) |

### How to Add a Parameter in the Dashboard UI

1. In the **Data** tab, write SQL with `:param_name` syntax
2. Databricks auto-detects the parameter and shows an input above the editor
3. On the **Canvas** tab, add a **Filter** widget:
   - Set **Type** to "Query parameter"
   - Select the parameter name
   - Choose a source dataset/column for the dropdown values
4. Now the filter widget drives the SQL parameter

### Parameter Behavior by Data Type

| Data Architecture | Parameter Experience |
| --- | --- |
| **Star Schema** | Parameters in WHERE clause — can filter on any joined dimension |
| **Gold Table** | Parameters in WHERE clause — simplest, filter on any column |
| **SQL View** | Parameters filter the view’s OUTPUT (can’t filter inside the view) |
| **Metric View** | Parameters in WHERE filter dimensions; or use YAML-level parameters for advanced logic |

---

## Part 4: Cross-Filtering — How Data Architecture Affects It

Cross-filtering lets viewers click a data point in one chart to filter all other charts on the same page.

### How It Works
- Click a bar in "Revenue by Region" → all other widgets filter to that region
- Works automatically for widgets with **matching column names** across datasets
- Enabled by default on: Bar, Line, Pie, Area, Scatter, Table widgets

### Critical Insight: Column Name Matching

Cross-filtering works by **matching column names** between datasets. This is where data architecture matters:

| Architecture | Cross-Filter Behavior |
| --- | --- |
| **Gold Table** | Best — all datasets have the same column names (e.g., `region`) |
| **Star Schema** | Works IF you alias consistently (e.g., always `AS region`) |
| **SQL Views** | Works if view output column names match other datasets |
| **Metric View** | Works on dimension names — but names may differ from other datasets |

### Best Practice for Cross-Filtering

**Use consistent column aliases across all datasets:**
```sql
-- Dataset 1 (Star Schema): revenue by region
SELECT r.region_name AS region, ...

-- Dataset 2 (Gold Table): orders by region
SELECT region, ...

-- Dataset 3 (Metric View): profit by region
SELECT `Region` AS region, ...
```

By aliasing everything to `region`, cross-filtering connects all three automatically.

### When Cross-Filtering Breaks
- Column names don’t match (`region_name` vs `region` vs `Region`)
- One dataset uses IDs (`region_id`) while others use names
- Metric view dimension names have spaces (`Product Category`) vs others (`product_category`)

**Fix:** Standardize aliases in your dashboard SQL. This is another advantage of Gold Tables — column names are pre-standardized.

---

In [0]:
%sql
-- ============================================================
-- ADVANCED: Date Range Parameters + Multiple Data Types
-- ============================================================
-- Combine date range parameters with different data sources
-- for a complete interactive dashboard experience.

-- Date-parameterized trend using Star Schema:
SELECT
  DATE_TRUNC('MONTH', f.order_date) AS order_month,
  r.region_name                     AS region,
  ROUND(SUM(f.net_revenue), 2)      AS total_revenue,
  COUNT(f.order_id)                 AS order_count
FROM fact_sales f
JOIN dim_region r ON f.region_id = r.region_id
WHERE f.order_date BETWEEN :start_date AND :end_date
GROUP BY DATE_TRUNC('MONTH', f.order_date), r.region_name
ORDER BY order_month, region;

-- Date-parameterized KPIs using Metric View:
-- SELECT
--   MEASURE(`Total Revenue`)    AS total_revenue,
--   MEASURE(`Total Profit`)     AS total_profit,
--   MEASURE(`Order Count`)      AS total_orders,
--   MEASURE(`Profit Margin`)    AS profit_margin,
--   MEASURE(`Unique Customers`) AS unique_customers
-- FROM mv_sales_metrics
-- WHERE `Order Date` BETWEEN :start_date AND :end_date
-- GROUP BY ALL;

## Part 5: Metric Views vs Dashboard Custom Calculations

Databricks dashboards support **calculated measures** and **calculated dimensions** defined directly in the widget configuration. How does this compare to metric views?

### Dashboard Custom Calculations
- Defined per-dataset in the dashboard UI
- Live only within that dashboard
- Quick ad-hoc metrics without modifying source data
- Power BI equivalent: DAX measures in a report

### Metric View Measures
- Defined centrally in Unity Catalog YAML
- Shared across ALL dashboards, Genie spaces, notebooks
- Governed, versioned, discoverable
- Power BI equivalent: DAX measures in a shared semantic model

### When to Use Each

| Scenario | Dashboard Calculation | Metric View |
| --- | --- | --- |
| Quick one-off metric for this dashboard | ✓ | |
| Metric needed across 5+ dashboards | | ✓ |
| Business-critical KPI (revenue, profit) | | ✓ |
| Experimental / exploratory analysis | ✓ | |
| Metric needs AI/Genie discoverability | | ✓ |
| Format metadata (currency, %) important | | ✓ |

### Recommended Workflow
1. **Start** with dashboard custom calculations for prototyping
2. **Promote** proven metrics to a metric view when they become shared/official
3. **Retire** dashboard calculations as the metric view becomes the source of truth

---

## Part 6: Complete Dashboard Architecture — Putting It All Together

### Recommended Multi-Page Dashboard Design

```
Page 1: Executive Summary
└─ Datasets: Metric View (consistent KPIs)
└─ Widgets: Counters (revenue, profit, orders, margin)
└─ Filters: Global date range parameter

Page 2: Regional Performance
└─ Datasets: Gold Table (fast, simple queries)
└─ Widgets: Bar chart (region), Stacked bar (region x channel)
└─ Filters: Page-level region multi-select

Page 3: Trend Analysis
└─ Datasets: SQL View (pre-aggregated monthly)
└─ Widgets: Line chart, Area chart (revenue + profit)
└─ Filters: Category parameter + date range

Page 4: Product Deep Dive
└─ Datasets: Star Schema (maximum flexibility)
└─ Widgets: Table (top products), Scatter (price vs volume)
└─ Filters: Dynamic drill-through from Page 2
```

### Why This Hybrid Works
- **Page 1 (Metric View):** Everyone agrees on what "revenue" means
- **Page 2 (Gold Table):** Sub-second response for high-traffic page
- **Page 3 (SQL View):** Pre-aggregated = no expensive GROUP BY at dashboard time
- **Page 4 (Star Schema):** Ad-hoc exploration without pre-building every view

### Interactivity Matrix

| Page | Filter Type | Data Source | Why |
| --- | --- | --- | --- |
| Executive | Global date param | Metric View | Consistent definitions + date flexibility |
| Regional | Page filter (region) | Gold Table | Simple cross-filtering on flat data |
| Trends | Parameter (category) | SQL View | Filter the view output dynamically |
| Products | Drill-through | Star Schema | Flexible JOINs for any product detail |

---

---

## Summary: Data Architecture for Interactive Dashboards

### Key Takeaways

| Data Architecture | Best For Interactivity | Watch Out For |
| --- | --- | --- |
| **Star Schema** | Full flexibility (any filter, any JOIN) | Column name consistency for cross-filtering |
| **Gold Table** | Seamless cross-filtering (flat data) | Must re-run ETL if new filter columns needed |
| **SQL View** | Pre-shaped data, simple parameters | Can only filter on output columns |
| **Metric View** | Consistent definitions + dimension filtering | MEASURE() syntax, must GROUP BY ALL |

### The Three Levels of Dashboard Interactivity

| Level | Feature | Implementation |
| --- | --- | --- |
| Basic | Filter widgets | Add filter widget, bind to dataset column |
| Intermediate | Parameters | Use `:param_name` in SQL, bind to filter widget |
| Advanced | Cross-filtering + Drill-through | Consistent column naming + multi-page design |

### Publishing Reminder

When your dashboard is ready:
1. Click **Publish** (top right)
2. Choose credential mode:
   - **Share data permission** — viewers use publisher's data access
   - **Individual data permission** — viewers use their own credentials
3. Published version is a frozen snapshot — updates require re-publishing

> A **companion Genie Space** is auto-created on publish. Viewers can ask natural language questions about the dashboard data — metric views are especially well-suited for this (synonyms, comments, format metadata).

### Final Recommendation

The strongest dashboards use a **hybrid approach**: metric views for consistent KPIs, gold tables for performance-critical pages, views for pre-shaped time-series, and star schemas for drill-through flexibility. Match the data architecture to the page’s purpose.